# Memory Engine Benchmarks

Shared pipeline for **LoCoMo**, **LongMemEval**, **HaluMem**, and **MemoryDocDataSet** using Ledger & Lens + Pulse/Atlas retrieval.

**Edit code in Python modules, not here:**
- `memory_retrieval.py` — retrieval + Atlas drill-down
- `benchmark_harness.py` — formation, Atlas, QA loop
- `dataset_locomo.py` / `dataset_longmemeval.py` / `dataset_halumem.py` / `dataset_memorydoc.py` — dataset adapters

Sections below: shared setup → LoCoMo → LongMemEval → HaluMem → MemoryDocDataSet.

## 0. Shared setup

In [51]:
import importlib
import os
import sys
from getpass import getpass
from pathlib import Path

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
NOTEBOOKS = ROOT / 'notebooks'
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))

# --- toggles ---
RUN_LOCOMO = False
RUN_LONGMEMEVAL = False
RUN_HALUMEM = False
RUN_MEMORYDOC = True

# --- models ---
GEMINI_MODEL = 'gemini-3.1-flash-lite'
SBERT_MODEL = 'all-MiniLM-L6-v2'
TEMPERATURE = 0.0
TOP_K = 12
MAX_CONTEXT_ITEMS = 24

# --- LoCoMo limits ---
LOCOMO_MAX_CONVERSATIONS = 20
LOCOMO_MAX_QUESTIONS = 500

# --- LongMemEval limits ---
LONGMEMEVAL_SPLIT = 'small'  # small | medium | oracle
LONGMEMEVAL_MAX_QUESTIONS = 100
LONGMEMEVAL_MAX_SESSIONS = None  # None = all sessions per haystack

# --- HaluMem limits ---
HALUMEM_SPLIT = 'medium'  # medium | long
HALUMEM_MAX_USERS = 10 # Max 20
HALUMEM_MAX_SESSIONS = None  # per user; None = all
HALUMEM_MAX_QUESTIONS = 300

# --- MemoryDocDataSet limits ---
MEMORYDOC_SPLIT = 'smoke'  # smoke | test | val | train | all
MEMORYDOC_MAX_WORLDS = 10
MEMORYDOC_MAX_QUESTIONS = 200
MEMORYDOC_MAX_DOC_CHARS = None  # truncate long docs for dev runs
MEMORYDOC_MAX_SESSIONS = None  # per micro-world

# --- paths ---
LOCOMO_DATA = ROOT / 'data' / 'locomo'
LONGMEMEVAL_DATA = ROOT / 'data' / 'longmemeval'
LOCOMO_RESULTS = LOCOMO_DATA / 'runs'
LONGMEMEVAL_RESULTS = LONGMEMEVAL_DATA / 'runs'
HALUMEM_DATA = ROOT / 'data' / 'halumem'
HALUMEM_RESULTS = HALUMEM_DATA / 'runs'
MEMORYDOC_DATA = ROOT / 'data' / 'memorydoc'
MEMORYDOC_RESULTS = MEMORYDOC_DATA / 'runs'

if not os.getenv('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('GEMINI_API_KEY: ')

print('ROOT:', ROOT)
print('RUN_LOCOMO:', RUN_LOCOMO, '| RUN_LONGMEMEVAL:', RUN_LONGMEMEVAL)
print('RUN_HALUMEM:', RUN_HALUMEM, '| RUN_MEMORYDOC:', RUN_MEMORYDOC)

ROOT: ..
RUN_LOCOMO: False | RUN_LONGMEMEVAL: False
RUN_HALUMEM: False | RUN_MEMORYDOC: True


In [52]:
import re
from google import genai
from google.genai import types

import memory_retrieval
import benchmark_harness
import dataset_locomo
import dataset_longmemeval
import dataset_halumem
import dataset_memorydoc

for mod in (memory_retrieval, benchmark_harness, dataset_locomo, dataset_longmemeval, dataset_halumem, dataset_memorydoc):
    importlib.reload(mod)

from memory_retrieval import MODULE_VERSION, RetrieverConfig
from benchmark_harness import (
    Ledger,
    build_episode_atlas,
    finalize_pipeline,
    parse_json_loose,
    run_benchmark,
    save_run,
    summarize_results,
)
from sentence_transformers import SentenceTransformer

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])


def gemini_text(system: str, user: str, *, json_mode: bool = False) -> str:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=TEMPERATURE,
            response_mime_type='application/json' if json_mode else None,
        ),
    )
    return (response.text or '').strip()


RETRIEVER_CFG = RetrieverConfig(top_k=TOP_K, max_context_items=MAX_CONTEXT_ITEMS, ground_via_atlas=True)
print('memory_retrieval version:', MODULE_VERSION)
print(gemini_text('Reply with one word.', 'Say hello.'))

memory_retrieval version: 20260802-benchmark-harness
Hello.


In [53]:
# Shared helpers

def build_full_pipeline(ledger: Ledger) -> benchmark_harness.MemoryPipeline:
    print('Loading SBERT:', SBERT_MODEL)
    embed_model = SentenceTransformer(SBERT_MODEL)
    print('Building Atlas episode summaries...')
    atlas = build_episode_atlas(ledger, gemini_text, parse_json_loose)
    print('Atlas entries:', len(atlas))
    compiled = benchmark_harness.compile_claims(ledger.extractions())
    print('Compiled claims:', len(compiled))
    stub = benchmark_harness.MemoryPipeline(ledger, compiled, atlas, None, None)
    pipeline = finalize_pipeline(stub, atlas, embed_model=embed_model, sbert_model=SBERT_MODEL, retriever_config=RETRIEVER_CFG)
    print('Index items:', len(pipeline.memory_index.items))
    return pipeline

## 1. LoCoMo

In [54]:
if RUN_LOCOMO:
    locomo_path = dataset_locomo.download_locomo(LOCOMO_DATA)
    locomo_records = dataset_locomo.load_records(locomo_path, LOCOMO_MAX_CONVERSATIONS)
    print(f'Loaded {len(locomo_records)} LoCoMo conversation(s)')

    locomo_ledger = Ledger()
    locomo_dia_maps = dataset_locomo.form_locomo(locomo_records, locomo_ledger, gemini_text=gemini_text, parse_json=parse_json_loose)
    print('Ledger:', len(locomo_ledger.deltas), 'deltas')

    locomo_pipeline = build_full_pipeline(locomo_ledger)

    locomo_questions, _ = dataset_locomo.build_questions(locomo_records, max_questions=LOCOMO_MAX_QUESTIONS)
    print('Questions:', len(locomo_questions))

    locomo_results = run_benchmark(
        locomo_pipeline,
        locomo_questions,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        evidence_fn=lambda qa, hits: dataset_locomo.evidence_in_hits(qa, hits, locomo_dia_maps),
    )

    locomo_summary = summarize_results(locomo_results)
    locomo_out = save_run(
        LOCOMO_RESULTS,
        benchmark='ledger_lens_locomo',
        architecture='ledger_and_lens+pulse_atlas_drilldown',
        config={
            'gemini_model': GEMINI_MODEL,
            'sbert_model': SBERT_MODEL,
            'memory_retrieval_version': MODULE_VERSION,
            'max_conversations': LOCOMO_MAX_CONVERSATIONS,
            'max_questions': LOCOMO_MAX_QUESTIONS,
            'ground_via_atlas': True,
        },
        pipeline=locomo_pipeline,
        summary=locomo_summary,
        results=locomo_results,
    )
    print('\n=== LoCoMo ===')
    print(f"Accuracy: {locomo_summary['gemini_judge_accuracy']:.1%}")
    print(f"Evidence in context: {locomo_summary['gold_evidence_in_context_rate']:.1%}")
    print('Saved ->', locomo_out)
else:
    print('Skipping LoCoMo (RUN_LOCOMO=False)')

Skipping LoCoMo (RUN_LOCOMO=False)


## 2. LongMemEval (cleaned)

Each question has its own haystack. Retrieval is scoped via `episode_prefix=lme:{question_id}:` so sessions from other questions never leak in.

**First run:** downloads `longmemeval_s_cleaned.json` from Hugging Face (~requires network).

Start with `LONGMEMEVAL_MAX_QUESTIONS=5` for a smoke test — formation is one Gemini extraction call per session.

In [55]:
if RUN_LONGMEMEVAL:
    lme_path = dataset_longmemeval.resolve_data_path(LONGMEMEVAL_DATA, LONGMEMEVAL_SPLIT)
    lme_records = dataset_longmemeval.load_records(lme_path, LONGMEMEVAL_MAX_QUESTIONS)
    print(f'Loaded {len(lme_records)} LongMemEval record(s) from', lme_path.name)

    lme_ledger = Ledger()
    dataset_longmemeval.form_longmemeval(
        lme_records,
        lme_ledger,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        max_sessions_per_question=LONGMEMEVAL_MAX_SESSIONS,
    )
    print('Ledger:', len(lme_ledger.deltas), 'deltas')

    lme_pipeline = build_full_pipeline(lme_ledger)
    lme_questions = dataset_longmemeval.build_questions(lme_records)

    lme_results = run_benchmark(
        lme_pipeline,
        lme_questions,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        evidence_fn=dataset_longmemeval.evidence_in_hits,
    )

    lme_summary = summarize_results(lme_results)
    lme_out = save_run(
        LONGMEMEVAL_RESULTS,
        benchmark='ledger_lens_longmemeval',
        architecture='ledger_and_lens+pulse_atlas_drilldown',
        config={
            'gemini_model': GEMINI_MODEL,
            'sbert_model': SBERT_MODEL,
            'memory_retrieval_version': MODULE_VERSION,
            'split': LONGMEMEVAL_SPLIT,
            'max_questions': LONGMEMEVAL_MAX_QUESTIONS,
            'ground_via_atlas': True,
        },
        pipeline=lme_pipeline,
        summary=lme_summary,
        results=lme_results,
    )
    print('\n=== LongMemEval ===')
    print(f"Accuracy: {lme_summary['gemini_judge_accuracy']:.1%}")
    print(f"Answer-session recall: {lme_summary['gold_evidence_in_context_rate']:.1%}")
    print('Saved ->', lme_out)
else:
    print('Skipping LongMemEval (RUN_LONGMEMEVAL=False)')

Skipping LongMemEval (RUN_LONGMEMEVAL=False)


## 3. HaluMem

Operation-level memory hallucination benchmark. This section runs the **QA task** over formed ledger memory and reports optional **extraction recall** vs reference memory points.

**First run:** downloads `HaluMem-Medium.jsonl` from Hugging Face (~140MB).

Start with `HALUMEM_MAX_USERS=1`, `HALUMEM_MAX_SESSIONS=3`, `HALUMEM_MAX_QUESTIONS=10` for smoke tests.

In [ ]:
if RUN_HALUMEM:
    halu_path = dataset_halumem.resolve_data_path(HALUMEM_DATA, HALUMEM_SPLIT)
    halu_users = dataset_halumem.load_users(halu_path, HALUMEM_MAX_USERS)
    print(f'Loaded {len(halu_users)} HaluMem user(s) from', halu_path.name)

    halu_ledger = Ledger()
    dataset_halumem.form_halumem(
        halu_users,
        halu_ledger,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        max_sessions_per_user=HALUMEM_MAX_SESSIONS,
    )
    print('Ledger:', len(halu_ledger.deltas), 'deltas')

    halu_extraction = dataset_halumem.score_extraction_recall(
        halu_users,
        halu_ledger,
        max_sessions_per_user=HALUMEM_MAX_SESSIONS,
    )
    print('Extraction recall:', f"{halu_extraction['memory_point_recall']:.1%}",
          f"({halu_extraction['memory_points_recalled']}/{halu_extraction['memory_points_total']} memory points)")

    halu_pipeline = build_full_pipeline(halu_ledger)
    halu_questions = dataset_halumem.build_questions(
        halu_users,
        max_questions=HALUMEM_MAX_QUESTIONS,
        max_sessions_per_user=HALUMEM_MAX_SESSIONS,
    )
    print('Questions:', len(halu_questions))

    halu_results = run_benchmark(
        halu_pipeline,
        halu_questions,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        evidence_fn=dataset_halumem.evidence_in_hits,
    )

    halu_summary = summarize_results(halu_results)
    halu_summary['extraction'] = halu_extraction
    halu_out = save_run(
        HALUMEM_RESULTS,
        benchmark='ledger_lens_halumem',
        architecture='ledger_and_lens+pulse_atlas_drilldown',
        config={
            'gemini_model': GEMINI_MODEL,
            'sbert_model': SBERT_MODEL,
            'memory_retrieval_version': MODULE_VERSION,
            'split': HALUMEM_SPLIT,
            'max_users': HALUMEM_MAX_USERS,
            'max_sessions_per_user': HALUMEM_MAX_SESSIONS,
            'max_questions': HALUMEM_MAX_QUESTIONS,
            'ground_via_atlas': True,
        },
        pipeline=halu_pipeline,
        summary=halu_summary,
        results=halu_results,
    )
    print('\n=== HaluMem ===')
    print(f"QA accuracy: {halu_summary['gemini_judge_accuracy']:.1%}")
    print(f"Evidence overlap: {halu_summary['gold_evidence_in_context_rate']:.1%}")
    print(f"Extraction recall: {halu_extraction['memory_point_recall']:.1%}")
    print('Saved ->', halu_out)
else:
    print('Skipping HaluMem (RUN_HALUMEM=False)')

Skipping HaluMem (RUN_HALUMEM=False)


## 4. MemoryDocDataSet

Joint conversational memory + long-document reasoning. Each micro-world has chat sessions **and** long documents; **Hybrid** questions require routing from conversation to document.

Default split `smoke` uses `data/memorydoc/fixtures/smoke_micro_world.json` until the official JSON release is placed under `data/memorydoc/`.

For full benchmark runs, download the official release (see [arxiv:2606.04442](https://arxiv.org/abs/2606.04442)) as `data/memorydoc/memorydoc_v1.json` and set `MEMORYDOC_SPLIT='all'`.

In [ ]:
if RUN_MEMORYDOC:
    mdoc_path = dataset_memorydoc.download_memorydoc(MEMORYDOC_DATA, MEMORYDOC_SPLIT)
    mdoc_worlds = dataset_memorydoc.load_micro_worlds(mdoc_path, MEMORYDOC_MAX_WORLDS)
    print(f'Loaded {len(mdoc_worlds)} micro-world(s) from', mdoc_path)

    mdoc_ledger = Ledger()
    mdoc_maps = dataset_memorydoc.form_memorydoc(
        mdoc_worlds,
        mdoc_ledger,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        max_doc_chars=MEMORYDOC_MAX_DOC_CHARS,
        max_sessions_per_world=MEMORYDOC_MAX_SESSIONS,
    )
    print('Ledger:', len(mdoc_ledger.deltas), 'deltas')

    mdoc_pipeline = build_full_pipeline(mdoc_ledger)
    mdoc_questions = dataset_memorydoc.build_questions(mdoc_worlds, max_questions=MEMORYDOC_MAX_QUESTIONS)
    print('Questions:', len(mdoc_questions))

    mdoc_results = run_benchmark(
        mdoc_pipeline,
        mdoc_questions,
        gemini_text=gemini_text,
        parse_json=parse_json_loose,
        evidence_fn=lambda qa, hits: dataset_memorydoc.evidence_in_hits(qa, hits, mdoc_maps),
    )

    mdoc_summary = summarize_results(mdoc_results)
    mdoc_summary['source_tag_accuracy'] = dataset_memorydoc.summarize_by_source_tag(mdoc_results)
    mdoc_out = save_run(
        MEMORYDOC_RESULTS,
        benchmark='ledger_lens_memorydoc',
        architecture='ledger_and_lens+pulse_atlas_drilldown',
        config={
            'gemini_model': GEMINI_MODEL,
            'sbert_model': SBERT_MODEL,
            'memory_retrieval_version': MODULE_VERSION,
            'split': MEMORYDOC_SPLIT,
            'max_worlds': MEMORYDOC_MAX_WORLDS,
            'max_questions': MEMORYDOC_MAX_QUESTIONS,
            'max_doc_chars': MEMORYDOC_MAX_DOC_CHARS,
            'ground_via_atlas': True,
        },
        pipeline=mdoc_pipeline,
        summary=mdoc_summary,
        results=mdoc_results,
    )
    print('\n=== MemoryDocDataSet ===')
    print(f"Accuracy: {mdoc_summary['gemini_judge_accuracy']:.1%}")
    print(f"Evidence in context: {mdoc_summary['gold_evidence_in_context_rate']:.1%}")
    print('By source tag:', mdoc_summary.get('source_tag_accuracy'))
    print('Saved ->', mdoc_out)
else:
    print('Skipping MemoryDocDataSet (RUN_MEMORYDOC=False)')

Loaded 1 micro-world(s) from ../data/memorydoc/fixtures/smoke_micro_world.json
Forming MemoryDoc world smoke-legal-01 ...
Ledger: 23 deltas
Loading SBERT: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building Atlas episode summaries...
Atlas entries: 5
Compiled claims: 15
Index items: 43
Questions: 5
✓ [Single-hop|fact] Which attorneys conducted the initial client consultation on the ...
   gold: Margaret Chen and David Rodriguez | pred: Margaret Chen and David Rodriguez
   retrieval: stage=fast drilled=20 ev=True
✓ [Single-hop|fact] What contractual clause did the Riverton case establish as unenfo...
   gold: Section 14(b) | pred: Section 14(b), which purported to waive all consequential damages without a reci
   retrieval: stage=cold drilled=21 ev=True
✓ [Single-hop|fact] What specific contractual clause did the precedent case Sarah Kim...
   gold: Non-compete clauses exceeding eighteen months for mid-level managers without tra | pred: Non-compete clauses exceeding eighteen months for mid-level managers without tra
   retrieval: stage=fast drilled=18 ev=True
✓ [Knowledge Update|fact] What was the legal team's recommended strategy as of March 2024?...
   gold: Lead with Section 14